# Preprocessing

## Importing Libraries

In [1]:
import pandas as pd
import numpy as np
from PIL import Image
import os
from sklearn.model_selection import train_test_split
import time

IMG_DIR = "../data/raw/images_training_rev1/images_training_rev1"
LABELS_PATH = "../data/raw/training_solutions_rev1/training_solutions_rev1.csv"

IMG_SIZE = 64

## Defining Class

In [2]:
labels_df = pd.read_csv(LABELS_PATH)
clean_df = labels_df[labels_df['Class1.3'] < 0.5].copy()

def assign_class(row):
    if row['Class1.1'] >= 0.5:
        return 'Smooth'
    elif row['Class4.1'] >= 0.5:
        return 'Spiral'
    else:
        return 'Irregular'

clean_df['label'] = clean_df.apply(assign_class, axis=1)
print(clean_df.shape)

(61534, 39)


## Testing Loading Function

### Testing loading on a Small Subset of 5 Images

In [ ]:
def load_and_resize(galaxy_id, img_dir=IMG_DIR, size=IMG_SIZE):
    path = os.path.join(img_dir, f"{galaxy_id}.jpg")
    img = Image.open(path).convert('RGB').resize((size, size))
    return np.array(img)

test_ids = clean_df['GalaxyID'].head(5).values
for gid in test_ids:
    arr = load_and_resize(gid)
    print(gid, arr.shape, arr.dtype)

100008 (64, 64, 3) uint8
100023 (64, 64, 3) uint8
100053 (64, 64, 3) uint8
100078 (64, 64, 3) uint8
100090 (64, 64, 3) uint8


### Testing loading on a Medium Subset of 2000 Images

In [4]:
subset_df = clean_df.sample(2000, random_state=0)

start = time.time()
X_subset = np.array([load_and_resize(gid) for gid in subset_df['GalaxyID']])
y_subset = subset_df['label'].values
elapsed = time.time() - start

print("Shape:", X_subset.shape)
print(f"Time for 2000 images: {elapsed:.2f} seconds")
print(f"Estimated time for full {len(clean_df)} images: {elapsed * len(clean_df) / 2000 / 60:.1f} minutes")

Shape: (2000, 64, 64, 3)
Time for 2000 images: 45.63 seconds
Estimated time for full 61534 images: 23.4 minutes


### Now Running it on the full Dataset (61k images)

In [5]:
start = time.time()
X = np.array([load_and_resize(gid) for gid in clean_df['GalaxyID']])
y = clean_df['label'].values
elapsed = time.time() - start

print("X shape:", X.shape)
print("y shape:", y.shape)
print(f"Total time: {elapsed/60:.1f} minutes")

X shape: (61534, 64, 64, 3)
y shape: (61534,)
Total time: 23.7 minutes


## Normalization - converting every value to be within 0 and 1

In [6]:
X_norm = X.astype('float32') / 255.0
print(X_norm.dtype, X_norm.min(), X_norm.max())

float32 0.0 1.0


## Labeling Class with numbers

In [7]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(le.classes_)
print(y_encoded[:10])

['Irregular' 'Smooth' 'Spiral']
[0 2 1 1 1 1 0 1 2 0]


##  train/val/test spliting

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X_norm, y_encoded, test_size=0.3, random_state=0, stratify=y_encoded
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=0, stratify=y_temp
)
#stratify=y_encoded / y_temp, makes sure that when it splits, the class propotions are preserved in both the output groups, matching the original datasets propotions.

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

Train: (43073, 64, 64, 3) Val: (9230, 64, 64, 3) Test: (9231, 64, 64, 3)


## Saving processed Arrays

In [9]:
np.save('../data/processed/X_train.npy', X_train)
np.save('../data/processed/X_val.npy', X_val)
np.save('../data/processed/X_test.npy', X_test)
np.save('../data/processed/y_train.npy', y_train)
np.save('../data/processed/y_val.npy', y_val)
np.save('../data/processed/y_test.npy', y_test)
print("Saved.")

Saved.
